# Train a Piper voice on your own recordings

Runs on a **free Colab GPU** (Runtime → Change runtime type → **T4 GPU**).

1. Record sentences with `recorder.html`, then run `prepare_piper_dataset.py` → you get a folder like `dataset_en` (with `wavs/` and `metadata.csv`). Zip it: `dataset_en.zip`.
2. Run the cells below in order. Training is saved to your **Google Drive**, so if Colab disconnects you can run everything again and it continues where it stopped.
3. At the end you download two files, `my_voice.onnx` and `my_voice.onnx.json`, and use them with
   `render_lesson.py --tts piper --voice-model my_voice.onnx`.

Fine-tuning starts from an existing Piper voice, so it learns your voice much faster than training from zero. Train English and Afrikaans as **separate** voices (run this notebook once per language).

## 1. Settings

In [ ]:
VOICE_NAME = "my_voice_en"        # change to e.g. "my_voice_af" for Afrikaans
LANGUAGE = "en"                    # "en" or "af"
ESPEAK_VOICE = {"en": "en-gb", "af": "af"}[LANGUAGE]
EXTRA_EPOCHS = 1000                # more = closer to your voice, but longer. 500-2000 is typical
BATCH_SIZE = 16                    # lower to 8 if you get "CUDA out of memory"

# Piper voice to start from (a medium-quality checkpoint). An English voice works
# as a starting point for Afrikaans too.
CHECKPOINT_PATTERN = r"^en/en_GB/[^/]+/medium/.+\.ckpt$"


## 2. Connect Google Drive (keeps your training safe if Colab disconnects)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
WORK = f"/content/drive/MyDrive/piper-voice/{VOICE_NAME}"
os.makedirs(WORK, exist_ok=True)
print("Saving to", WORK)

## 3. Install Piper training tools (takes a few minutes)

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build espeak-ng > /dev/null
%cd /content
!test -d piper1-gpl || git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace > /dev/null
%cd /content

## 4. Upload your dataset zip (made by `prepare_piper_dataset.py`)

In [ ]:
import zipfile, glob
DATA = f"{WORK}/dataset"
if not os.path.exists(f"{DATA}/metadata.csv"):
    from google.colab import files
    uploaded = files.upload()                      # choose dataset_en.zip
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content/unzipped")
    meta = glob.glob("/content/unzipped/**/metadata.csv", recursive=True)[0]
    os.makedirs(DATA, exist_ok=True)
    !cp -r "{os.path.dirname(meta)}/." "{DATA}/"
n = sum(1 for _ in open(f"{DATA}/metadata.csv", encoding="utf-8"))
print(n, "clips ready in", DATA)

## 5. Download a starting checkpoint

In [ ]:
import re
from huggingface_hub import list_repo_files, hf_hub_download
START = f"{WORK}/start.ckpt"
if not os.path.exists(START):
    files_ = [f for f in list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset")
              if re.match(CHECKPOINT_PATTERN, f)]
    print("Matching checkpoints:", *files_, sep="\n  ")
    path = hf_hub_download("rhasspy/piper-checkpoints", files_[0], repo_type="dataset")
    !cp "{path}" "{START}"
    print("Using", files_[0])
print("Start checkpoint:", START)

## 6. Train

If Colab disconnects, run cells 1-5 again, then this one: it continues from the last checkpoint saved in your Drive.

In [ ]:
import torch
ckpts = sorted(glob.glob(f"{WORK}/lightning_logs/**/checkpoints/*.ckpt", recursive=True), key=os.path.getmtime)
resume = ckpts[-1] if ckpts else START
start_epoch = torch.load(START, map_location="cpu", weights_only=False).get("epoch", 0)
max_epochs = start_epoch + EXTRA_EPOCHS
print(f"Continuing from {resume}; training until epoch {max_epochs}")

!python3 -m piper.train fit \
  --data.voice_name "{VOICE_NAME}" \
  --data.csv_path "{DATA}/metadata.csv" \
  --data.audio_dir "{DATA}/wavs" \
  --model.sample_rate 22050 \
  --data.espeak_voice "{ESPEAK_VOICE}" \
  --data.cache_dir "/content/cache" \
  --data.config_path "{WORK}/config.json" \
  --data.batch_size {BATCH_SIZE} \
  --trainer.max_epochs {max_epochs} \
  --trainer.default_root_dir "{WORK}" \
  --ckpt_path "{resume}"
# If a flag is rejected, see the options with:  !python3 -m piper.train fit --help

## 7. Export your voice and download it

In [ ]:
ckpts = sorted(glob.glob(f"{WORK}/lightning_logs/**/checkpoints/*.ckpt", recursive=True), key=os.path.getmtime)
latest = ckpts[-1]
print("Exporting", latest)
!python3 -m piper.train.export_onnx --checkpoint "{latest}" --output-file "{WORK}/{VOICE_NAME}.onnx"
!cp "{WORK}/config.json" "{WORK}/{VOICE_NAME}.onnx.json"

# Quick listen test
!echo "Hello, this is my voice. Today we will learn about stationary points." | python3 -m piper -m "{WORK}/{VOICE_NAME}.onnx" -f /content/test.wav
from IPython.display import Audio, display
display(Audio("/content/test.wav"))

from google.colab import files
files.download(f"{WORK}/{VOICE_NAME}.onnx")
files.download(f"{WORK}/{VOICE_NAME}.onnx.json")